# AML - SAIGE-GENE SET-BASED ANALYSIS - FIL NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [14]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261102 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# COMMAND 

In [ ]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do

    echo "Processing population: ${pop}"
    
    for pheno in aml_phenotype; do 

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/EUR_deprivation_index_file.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVarRatio=TRUE \
                --IsOverwriteVarianceRatioFile=TRUE \
                --isCateVarianceRatio=TRUE \
                --phenoCol=aml_phenotype \
                --traitType=binary \
                --covarColList="age,sex,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10" \
                --qCovarColList=sex \
                --sampleIDColinphenoFile=person_id \
                --outputPrefix="${output}/${pop}_${pheno}_restricted_deprivation_cohort_null_glmm" 

    done
done


Overwriting saige_gene_step01.sh


# SUBMIT COMMAND 

In [22]:
%%bash --out saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --preemptible \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --script "saige_gene_step01.sh"


Job properties:
  job-id: saige-gene--vienose--261103-213213-84
  job-name: saige-gene-step01
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/saige-gene--vienose--261103-213213-84-0-0
Launched job-id: saige-gene--vienose--261103-213213-84
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261103-213213-84' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261103-213213-84' --users 'vienose'


In [31]:
!dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261103-213213-84' --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
saige-gene-s...  SUCCESS   11-03 21:42:00

